# E-commerce Customer Segmentation & Retention Opportunity Analysis

## 1. Project Overview

### Dataset Context

This project uses the **Online Retail II UCI** dataset hosted on **Kaggle**.

Dataset source: [Kaggle - Online Retail II UCI](https://www.kaggle.com/datasets/mashlyn/online-retail-ii-uci)

The dataset contains transaction-level records from an anonymized UK-based non-store online retail business between **December 2009 and December 2011**. The company mainly sells all-occasion giftware products, and many of its customers are wholesalers.

Each row represents a product line within an invoice, including information such as invoice number, product code, product description, quantity, invoice date, unit price, customer ID, and country.

### Business Context

For an online retail business, not all customers contribute equally to revenue. Some customers purchase frequently and generate high revenue, while others may buy once and never return.

Because of this, treating all customers the same can lead to inefficient CRM and retention efforts.

This project uses customer transaction data to identify customer segments based on purchase behavior and suggest practical CRM actions for each segment.

### Main Business Question

Which customer segments should the business prioritize for retention, reactivation, and revenue growth?

### Analytical Approach

This project uses a simple and interpretable **RFM segmentation** approach:

- **Recency**: How recently a customer purchased
- **Frequency**: How often a customer purchased
- **Monetary**: How much revenue a customer generated

The goal is to translate transaction data into customer segments such as high-value customers, loyal customers, new customers, at-risk customers, and inactive customers.

### Expected Output

The final output will include:

- A cleaned transaction dataset
- Customer-level RFM metrics
- Customer segments
- Segment-level performance summary
- CRM recommendations for retention, reactivation, and revenue growth

## 2. Setup and Data Loading

This section imports the required Python libraries and loads the transaction dataset for initial inspection.

In [2]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# Local setup: set the project directory

os.chdir(r"D:\bell\DATA ANALYST PERSONAL PROJECT\data-portfolio\04_E-commerce Customer Segmentation & Retention Opportunity Analysis")

df = pd.read_csv("data/online_retail_II.csv", encoding="ISO-8859-1")

df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


### Initial Preview

The dataset was loaded successfully. Each row represents one product line within an invoice, including product information, transaction quantity, invoice date, unit price, customer ID, and country.

## 3. Initial Data Inspection

This section checks the dataset structure and identifies the main data quality issues before cleaning.

### 3.1 Dataset Size

First, I check the number of rows and columns in the dataset.

In [3]:
basic_structure = pd.DataFrame({
    "Metric": ["Rows", "Columns"],
    "Value": [df.shape[0], df.shape[1]]
})

basic_structure

,Metric,Value
0,Rows,1067371
1,Columns,8


### 3.2 Column Overview

Next, I check each column's data type and missing values.

In [4]:
column_summary = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Missing Values": df.isna().sum().values,
    "Missing %": (df.isna().mean() * 100).round(2).values
})

column_summary

,Column,Data Type,Missing Values,Missing %
0,Invoice,object,0,0.00
1,StockCode,object,0,0.00
2,Description,object,4382,0.41
3,Quantity,int64,0,0.00
4,InvoiceDate,object,0,0.00
5,Price,float64,0,0.00
6,Customer ID,float64,243007,22.77
7,Country,object,0,0.00


### 3.3 Key Data Quality Checks

Before cleaning, I check duplicate rows, cancelled invoices, and non-positive quantity or price values.

In [5]:
data_quality_checks = pd.DataFrame({
    "Check": [
        "Duplicate rows",
        "Cancelled invoices",
        "Rows with Quantity <= 0",
        "Rows with Price <= 0"
    ],
    "Count": [
        df.duplicated().sum(),
        df["Invoice"].astype(str).str.startswith("C").sum(),
        (df["Quantity"] <= 0).sum(),
        (df["Price"] <= 0).sum()
    ]
})

data_quality_checks

,Check,Count
0,Duplicate rows,34335
1,Cancelled invoices,19494
2,Rows with Quantity <= 0,22950
3,Rows with Price <= 0,6207


### 3.4 Numeric Summary

I review the basic statistics of quantity and price to understand their ranges before cleaning.

In [6]:
df[["Quantity", "Price"]].describe().round(2)

,Quantity,Price
count,1067371.00,1067371.00
mean,9.94,4.65
std,172.71,123.55
min,-80995.00,-53594.36
25%,1.00,1.25
50%,3.00,2.10
75%,10.00,4.15
max,80995.00,38970.00


### Initial Inspection Summary

The dataset contains 1,067,371 rows and 8 columns. The main data quality issues are missing customer IDs, duplicate rows, cancelled invoices, non-positive quantity values, and non-positive price values.

These issues need to be handled before building customer-level RFM metrics because the analysis requires valid customer IDs and valid purchase transactions. The `InvoiceDate` column will also be converted to datetime format in the cleaning step.

## 4. Data Cleaning

This section prepares the dataset for customer-level RFM analysis.

For this project, valid sales transactions are defined as records that:

- Have a valid customer ID
- Are not cancelled invoices
- Have positive quantity
- Have positive price
- Are not duplicate rows

These rules are used because RFM segmentation requires valid customer-level purchase transactions.

### 4.1 Create a Clean Copy

I create a separate cleaned dataframe so the original dataset remains unchanged.

In [7]:
df_clean = df.copy()

### 4.2 Apply Cleaning Rules

I remove records that are not suitable for customer-level purchase analysis.

In [9]:
rows_before = df_clean.shape[0]

df_clean = df_clean.dropna(subset=["Customer ID"])
df_clean = df_clean[~df_clean["Invoice"].astype(str).str.startswith("C")]
df_clean = df_clean[df_clean["Quantity"] > 0]
df_clean = df_clean[df_clean["Price"] > 0]
df_clean = df_clean.drop_duplicates()

rows_after = df_clean.shape[0]

rows_before, rows_after

(1067371, 779425)

### 4.3 Convert Date Column

The invoice date column is converted to datetime format so it can be used later to calculate recency.

In [10]:
df_clean["InvoiceDate"] = pd.to_datetime(df_clean["InvoiceDate"])

df_clean["InvoiceDate"].min(), df_clean["InvoiceDate"].max()

(Timestamp('2009-12-01 07:45:00'), Timestamp('2011-12-09 12:50:00'))

### 4.4 Create Revenue Column

Revenue is calculated at the invoice-line level using quantity and unit price.

In [11]:
df_clean["Revenue"] = df_clean["Quantity"] * df_clean["Price"]

df_clean[["Quantity", "Price", "Revenue"]].head()

,Quantity,Price,Revenue
0,12,6.95,83.4
1,12,6.75,81.0
2,12,6.75,81.0
3,48,2.10,100.8
4,24,1.25,30.0


### 4.5 Cleaning Summary

I compare the number of rows before and after cleaning to confirm how many records were removed.

In [12]:
cleaning_summary = pd.DataFrame({
    "Stage": ["Before cleaning", "After cleaning", "Rows removed"],
    "Rows": [rows_before, rows_after, rows_before - rows_after]
})

cleaning_summary

,Stage,Rows
0,Before cleaning,1067371
1,After cleaning,779425
2,Rows removed,287946


### 4.6 Post-Cleaning Checks

I check the cleaned dataset to confirm that the main data quality issues have been handled.

In [13]:
post_cleaning_checks = pd.DataFrame({
    "Check": [
        "Missing Customer ID",
        "Cancelled invoices",
        "Rows with Quantity <= 0",
        "Rows with Price <= 0",
        "Duplicate rows"
    ],
    "Count": [
        df_clean["Customer ID"].isna().sum(),
        df_clean["Invoice"].astype(str).str.startswith("C").sum(),
        (df_clean["Quantity"] <= 0).sum(),
        (df_clean["Price"] <= 0).sum(),
        df_clean.duplicated().sum()
    ]
})

post_cleaning_checks

,Check,Count
0,Missing Customer ID,0
1,Cancelled invoices,0
2,Rows with Quantity <= 0,0
3,Rows with Price <= 0,0
4,Duplicate rows,0


### Cleaning Summary Interpretation

After cleaning, the dataset contains 779,425 valid purchase transaction lines. The cleaning process removed records with missing customer IDs, cancelled invoices, non-positive quantity or price values, and duplicate rows.

The post-cleaning checks confirm that these issues have been handled. This cleaned dataset is now suitable for customer-level RFM analysis.